# FlyRank Search Ranking & Discoverability Capstone

**Lane:** Refresh / Content Opportunity Scoring
**Goal:** Score pages that are growing, declining, recovering, or worth review, and output a ranked action engine with reason codes.

## 1. Setup and Authentication
We use DuckDB to directly query the Hugging Face dataset. Note: You need a valid Hugging Face read token.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set up Hugging Face token (replace 'YOUR_TOKEN' or set in environment variables)
hf_token = os.environ.get('HF_TOKEN', 'YOUR_TOKEN_HERE')
os.environ['HF_TOKEN'] = hf_token

## 2. Data Extraction
Connect to DuckDB and query `hf://datasets/FlyRank/internship-warehouse`.

In [ ]:
conn = duckdb.connect(':memory:')
conn.execute("INSTALL httpfs; LOAD httpfs;")

# In a real execution, we would query the dataset directly:
# df = conn.execute("""
#     SELECT 
#         url_hash,
#         SUM(clicks) as total_clicks,
#         SUM(impressions) as total_impressions,
#         AVG(position) as avg_position,
#         date
#     FROM 'hf://datasets/FlyRank/internship-warehouse/*.parquet'
#     WHERE date >= '2023-01-01'
#     GROUP BY url_hash, date
# """).df()

# For this notebook demonstration without a token, we create synthetic mock data reflecting the schema
np.random.seed(42)
n_urls = 1000
mock_data = pd.DataFrame({
    'url_hash': [f'hash_{i}' for i in range(n_urls)],
    'impressions_current': np.random.randint(100, 10000, n_urls),
    'clicks_current': np.random.randint(1, 1000, n_urls),
    'position_current': np.random.uniform(1.0, 50.0, n_urls),
    'impressions_previous': np.random.randint(100, 10000, n_urls),
    'clicks_previous': np.random.randint(1, 1000, n_urls),
    'position_previous': np.random.uniform(1.0, 50.0, n_urls),
})

mock_data['ctr_current'] = mock_data['clicks_current'] / mock_data['impressions_current']
mock_data['ctr_previous'] = mock_data['clicks_previous'] / mock_data['impressions_previous']

# Feature engineering: growth indicators
mock_data['clicks_diff'] = mock_data['clicks_current'] - mock_data['clicks_previous']
mock_data['impressions_diff'] = mock_data['impressions_current'] - mock_data['impressions_previous']
mock_data['position_diff'] = mock_data['position_previous'] - mock_data['position_current'] # positive is better

mock_data.head()

## 3. Modeling / Opportunity Scoring
We model the "Opportunity Score" based on the momentum (e.g., declining clicks but stable impressions = high refresh opportunity).

In [ ]:
# Target: 'opportunity_score'
# We synthesize a target label for training where high impressions but declining clicks and position indicates high opportunity.
mock_data['opportunity_score'] = (
    np.log1p(mock_data['impressions_current']) * 0.5 - 
    (mock_data['clicks_diff'] / mock_data['clicks_previous']) * 20 - 
    mock_data['position_diff'] * 0.5
)
mock_data['opportunity_score'] = (mock_data['opportunity_score'] - mock_data['opportunity_score'].min()) / (mock_data['opportunity_score'].max() - mock_data['opportunity_score'].min())

features = ['impressions_current', 'position_current', 'ctr_current', 'clicks_diff', 'impressions_diff', 'position_diff']
X = mock_data[features]
y = mock_data['opportunity_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"R2: {r2_score(y_test, y_pred):.4f}")

## 4. Ranked Recommendations & Reason Codes
We apply the model to score all content and generate reason codes based on feature contributions.

In [ ]:
mock_data['predicted_opportunity'] = model.predict(X)

def generate_reason(row):
    if row['clicks_diff'] < -50 and row['impressions_diff'] > 0:
        return 'Declining CTR despite stable visibility (Rewrite Title/Meta)'
    elif row['position_diff'] < -3:
        return 'Losing Rank Momentum (Content Refresh Needed)'
    elif row['impressions_current'] > 5000 and row['ctr_current'] < 0.02:
        return 'High Visibility, Low Engagement (Optimize Intent)'
    else:
        return 'Monitor / Maintain'

mock_data['reason_code'] = mock_data.apply(generate_reason, axis=1)
ranked_recommendations = mock_data.sort_values(by='predicted_opportunity', ascending=False)

ranked_recommendations[['url_hash', 'predicted_opportunity', 'reason_code']].head(10)